# Lab 3 · Build the `Value` autograd engine — the **forward** pass
### Lecture 3 · *Computational graphs & the maths of AI* · 2026-08-05

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsrobinson/me324/blob/main/labs/lab-03-value-forward.ipynb)

**How this notebook works.** Cells marked `# TODO` are for you to fill in — each one is
short and clearly marked. Worked answers for all of them are in the **Solutions** section
at the very bottom, so you will never be stuck for long. Have a real go first, then check.

**And please use an LLM.** ChatGPT, Claude, Gemini, DeepSeek — whichever you like.
*"In Python, how do I …?"* is exactly the kind of question these tools are excellent at, and
looking things up this way is what every working researcher does. Two habits worth keeping:
ask for the **explanation** rather than just the line, and **run everything** it hands you.
The exam is closed-book, so what counts is that you can read the code back and say what it
does.

---

#### Today's goals
By the end of this lab you will be able to:

1. Explain what a **computational graph** is, and how a *forward pass* walks it left-to-right (Lecture 3).
2. Build a `Value` class whose arithmetic (`+`, `*`, `**`, `exp`, `log`, `relu`) **records the graph as you compute**.
3. Understand the key idea — every operation stores its **local derivative** with respect to each child — that will make backpropagation almost free *tomorrow*.
4. Visualise an expression such as `c = a * b + 1` as a **directed acyclic graph (DAG)**.

> **Where this is going (one class, three labs).** This is Andrej Karpathy's `micrograd`, walked slowly.
> **Today (Lab 3)** we build only the *forward* parts of `Value`.
> **Tomorrow (Lab 4)** we add `backward()` and the *same class* starts to **learn**.
> Much later, in **Lab 11**, this very class powers a from-scratch GPT. Keep the class you build today — you will not throw it away.

## ⏱️ Plan for today (~90 minutes)

This lab is built for a single 90-minute session, and it's **completely fine not to finish every cell in the room**.

- **Core — do these:** building the forward `Value` class: `__add__`, `__mul__`, `__pow__` (Steps 1–3).
- **Stretch / take-home — skip if short on time:** `exp`/`log`/`relu` and the graph visualiser — nice-to-have, not essential today.

_Most of the code is written for you; the `# TODO` cells are the parts you write. Worked answers are in the **Solutions** section at the bottom._

### How to work through this notebook

- Read the **markdown** between code cells — it explains every step. The pacing is deliberately slow.
- Run cells **top to bottom** (`Shift`+`Enter`). Each cell depends on the ones above it.
- 🐍 Pure Python today: **no PyTorch, no GPU**. We are building automatic differentiation from scratch so that the frameworks you meet later feel like old friends.
- Look out for the ⭐ — it marks the single most important idea in the lab.

In [ ]:
# ---- Run me first ----
# Lab 3 needs nothing but Python's standard library.
import math   # for exp() and log() later

print("Ready. No PyTorch, no GPU — just Python. Let's build autodiff from scratch.")

## Recap: a calculation is a *graph* (from Lecture 3)

In the lecture we saw that any calculation can be drawn as a **computational graph** — a flow-chart of the arithmetic:

- **Value nodes** (boxes) hold *data* — and will later also hold a *gradient* (their slot for $\frac{\partial L}{\partial\,\cdot}$).
- **Operation nodes** apply a function (`+`, `*`, `**`, …) to incoming values.
- The graph is a **DAG** (directed, acyclic): arrows only ever flow forward.

We also previewed the **two passes**:

| | direction | what happens |
|---|---|---|
| **Forward** | left → right | at each node, compute its output from its inputs; remember it |
| **Backward** | right → left | at each node, use the **chain rule** to get $\frac{\partial L}{\partial\,\text{node}}$ |

And the chain rule itself — the engine of the backward pass:

$$\frac{dy}{dx} = \frac{dy}{du}\cdot\frac{du}{dx}.$$

**Today we build the forward pass and the graph.** But we do one clever extra thing while going forward, which makes tomorrow's backward pass almost trivial. That clever thing is the heart of this lab.

## What we are building: the `Value` class

A `Value` wraps a single number and behaves like one — you can write `a * b + 1` with `Value`s and it just works. The twist is that, as you do arithmetic, each `Value` quietly remembers **where it came from**.

Every `Value` stores four things:

| attribute | meaning |
|---|---|
| `data` | the number itself (e.g. `2.0`) |
| `grad` | its gradient — **stays `0` today**; tomorrow's `backward()` fills it in |
| `_children` | the `Value`(s) this one was built from (its inputs in the graph) |
| `_local_grads` | the **local derivative** of this node's output w.r.t. each child |

We will build it up gently:

1. First a tiny sketch that can only **add** — just to see the machinery.
2. Then the ⭐ central idea using **multiplication**.
3. Then the complete forward class: `*`, `**`, `exp`, `log`, `relu`, the convenience operators, and a `backward()` **stub** for tomorrow.
4. Then we test each operation, visualise a graph, and check our numbers against plain Python.

## Step 1 — a first sketch: a `Value` that can add

Let's start as small as possible: a `Value` that stores a number, and an `__add__` so we can write `a + b`.

When Python sees `a + b`, it calls `a.__add__(b)`. Our version returns a **new** `Value` whose:

- `data` is `a.data + b.data`,
- `_children` are `(a, b)` — remembering what made it,
- `_local_grads` are `(1, 1)` — because, by the **sum rule** from the lecture,

$$\frac{\partial (a + b)}{\partial a} = 1 \qquad\text{and}\qquad \frac{\partial (a + b)}{\partial b} = 1.$$

(That `if isinstance(...)` line just lets us write `a + 3` by wrapping the plain `3` in a `Value` first.)

In [ ]:
# A first sketch — just enough to ADD two Values and record the graph.
class ValueSketch:
    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0                    # gradient slot (filled in tomorrow)
        self._children = children        # the Values this one was built from
        self._local_grads = local_grads  # d(self)/d(child) for each child

    def __add__(self, other):
        other = other if isinstance(other, ValueSketch) else ValueSketch(other)
        return ValueSketch(self.data + other.data, (self, other), (1, 1))

    def __repr__(self):
        return f"ValueSketch(data={self.data})"

# Try it:
a = ValueSketch(2.0)
b = ValueSketch(3.0)
c = a + b
print("c.data         =", c.data)          # 5.0  -> the forward value
print("c._children    =", c._children)     # (a, b) -> the graph edges
print("c._local_grads =", c._local_grads)  # (1, 1) -> local derivatives

Look at what just happened. The line `c = a + b` did **two** jobs at once:

- it computed the **value** `5.0`, and
- it recorded the **graph**: `c` knows its children are `a` and `b`, and that the output changes by `1` for a unit change in either child.

That second job — storing `_children` and `_local_grads` — is the whole point. It is what turns ordinary arithmetic into a differentiable program.

## ⭐ Step 2 — store the *local* gradient of every operation

When we compute `out = a * b`, we obviously want the **value** `a.data * b.data`. But we *also* record, on the new node, **how sensitive the output is to each input** — the *local* derivative of the output with respect to each child:

$$\frac{\partial (ab)}{\partial a} = b \qquad\text{and}\qquad \frac{\partial (ab)}{\partial b} = a.$$

So for multiplication the two local gradients are **`(b.data, a.data)`** — note the **swap**: the derivative with respect to `a` is *`b`'s* value, and the derivative with respect to `b` is *`a`'s* value.

We store them next to the children, in the same order:

```python
return Value(a.data * b.data, children=(a, b), local_grads=(b.data, a.data))
```

**Why record this now?** The forward pass is the *only* moment when `a` and `b` are calculated. By saving the local derivative in the forward pass, we don't have to re-access these values.

Here are the local gradients for *every* operation we will implement (same as Lecture 3's derivative cheat-sheet):

| operation | output | local grad(s) w.r.t. each child |
|---|---|---|
| `a + b` | `a + b` | `(1, 1)` — sum rule |
| `a * b` | `a * b` | `(b, a)` — **the swap** |
| `a ** n` | `a ** n` | `(n * a ** (n-1),)` — power rule |
| `a.exp()` | `e**a` | `(e**a,)` |
| `a.log()` | `ln a` | `(1 / a,)` |
| `a.relu()` | `max(0, a)` | `(1 if a > 0 else 0,)` |

## Step 3 — the complete forward `Value` class

Now we can write the real class. It adds `*`, `**`, `exp`, `log`, `relu`, and a set of **convenience operators** (`-`, subtraction, division, …) that are *all defined in terms of the operations above*, so we only have to get the core ones right.

It also includes a `backward()` **stub**. We will not write this today — the stub is just temporary.

> One small thing you can ignore: `__slots__` is a minor performance tweak that declares the only four attributes a `Value` is allowed to have. It is there so the class is identical to the one we reuse in Lab 11; it does not change how you use it.

In [ ]:
import math

class Value:
    """A scalar node in a computation graph (micrograd style).

    Records its data, a grad slot (filled in TOMORROW by backward()), its
    children, and the LOCAL derivative of this node w.r.t. each child.
    """
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0                     # gradient — stays 0 until Lab 4's backward()
        self._children = children         # the Value(s) this node was built from
        self._local_grads = local_grads   # d(self)/d(child), one per child

    # ---------- core operations (each builds the graph as it computes) ----------

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # sum rule: d(a + b)/da = 1, d(a + b)/db = 1
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out_data = self.data * other.data
        # TODO: fill in the LOCAL derivatives of a * b.
        #   d(a*b)/da = ?   and   d(a*b)/db = ?   (see the ⭐ worked example above)
        local_grads = (1.0, 1.0)          # <-- FIX ME (currently wrong on purpose)
        return Value(out_data, (self, other), local_grads)

    def __pow__(self, other):
        # `other` is a plain number (the exponent), e.g.  x ** 2
        out_data = self.data ** other
        # TODO: fill in the LOCAL derivative using the power rule:
        #   d(x ** n)/dx = n * x ** (n - 1)
        local_grads = (1.0,)              # <-- FIX ME (currently wrong on purpose)
        return Value(out_data, (self,), local_grads)

    def exp(self):
        # d(e**x)/dx = e**x
        return Value(math.exp(self.data), (self,), (math.exp(self.data),))

    def log(self):
        # natural log; d(ln x)/dx = 1 / x
        return Value(math.log(self.data), (self,), (1 / self.data,))

    def relu(self):
        # relu(x) = max(0, x); slope is 1 where x > 0, else 0
        return Value(max(0, self.data), (self,), (float(self.data > 0),))

    # ---------- convenience operators (given — all built on the ops above) ----------
    def __neg__(self):             return self * -1
    def __radd__(self, other):     return self + other
    def __sub__(self, other):      return self + (-other)
    def __rsub__(self, other):     return other + (-self)
    def __rmul__(self, other):     return self * other
    def __truediv__(self, other):  return self * other ** -1
    def __rtruediv__(self, other): return other * self ** -1
    def __repr__(self):            return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # ---------- backward() is built TOMORROW (Lab 4): friendly stub for now ----------
    def backward(self):
        raise NotImplementedError(
            "backward() is built TOMORROW in Lab 4 — that is the step that makes the "
            "network LEARN. Today we only build the forward graph.")

print("Defined Value (forward only). Fix the two TODOs — the answers are in the Solutions section at the bottom.")

## Step 4 — exercise each operation and inspect the graph

Now we can use the class to perform operations. For each, we print the resulting
`.data` (the forward value) and inspect `._children` and `._local_grads` so we can
*see* the graph being recorded.

**Addition** — local grads should be `(1, 1)`.

In [ ]:
a = Value(2.0)
b = Value(3.0)
c = a + b
print("c              =", c)
print("c.data         =", c.data)           # expect 5.0
print("c._children    =", c._children)       # the two inputs a, b
print("c._local_grads =", c._local_grads)    # expect (1, 1)
assert c.data == 5.0
print("OK: addition forward value is correct.")

**Multiplication.** The local grads should be swapped versions of the values `(b.data, a.data)`.

In [ ]:
a = Value(2.0)
b = Value(3.0)
d = a * b
print("d              =", d)
print("d.data         =", d.data)            # expect 6.0
print("d._children    =", d._children)
print("d._local_grads =", d._local_grads)    # SHOULD be (3.0, 2.0) = (b.data, a.data)

assert d.data == 6.0, "forward value of a*b should be 6.0"

expected = (b.data, a.data)
if d._local_grads == expected:
    print(f"\u2705 local grads correct: {d._local_grads} == (b.data, a.data) = {expected}")
else:
    print(f"\u274c local grads not right yet: got {d._local_grads}, want {expected}.")
    print("   Fix the TODO in __mul__ (see Solutions, at the bottom), then re-run this.")

**Power** — for `x ** n` the local grad is `(n * x ** (n-1),)` (power rule).

In [ ]:
x = Value(2.0)
p = x ** 3
print("p              =", p)
print("p.data         =", p.data)            # expect 8.0
print("p._children    =", p._children)
print("p._local_grads =", p._local_grads)    # SHOULD be (3 * 2**2,) = (12.0,)

assert p.data == 8.0, "forward value of 2**3 should be 8.0"

expected = (3 * x.data ** 2,)
if p._local_grads == expected:
    print(f"\u2705 local grad correct: {p._local_grads} == (n*x**(n-1),) = {expected}")
else:
    print(f"\u274c local grad not right yet: got {p._local_grads}, want {expected}.")
    print("   Fix the TODO in __pow__ (see Solutions, at the bottom), then re-run this.")

**`exp`, `log`, `relu`** — three more nodes, each with one child and one local grad.

In [ ]:
e  = Value(1.0).exp()
l  = Value(2.0).log()
rn = Value(-3.0).relu()
rp = Value(4.0).relu()

print("exp(1)  ->", e,  " local:", e._local_grads)    # data e≈2.718, local e≈2.718
print("log(2)  ->", l,  " local:", l._local_grads)    # data ≈0.693, local 1/2 = 0.5
print("relu(-3)->", rn, " local:", rn._local_grads)   # data 0.0,  local 0.0
print("relu(4) ->", rp, " local:", rp._local_grads)   # data 4.0,  local 1.0

assert abs(e.data - math.e) < 1e-9
assert abs(l.data - math.log(2.0)) < 1e-9
assert rn.data == 0.0 and rp.data == 4.0
print("OK: exp / log / relu forward values are correct.")

## Step 5 — drawing the graph

Finally, let's *see* a computation graph, mirroring the pictures from Lecture 3.

Our helper walks `._children` from a root node and draws the DAG. We use **graphviz** for a
nice picture if it is available (it is, on Colab), and fall back to a plain-text tree otherwise —
so this cell is always safe to run.

Each **box** is a `Value` (showing its `data` and `grad`). Each **edge** is labelled with the
**local gradient** `d(parent)/d(child)` — i.e. the numbers we stored on the forward pass and that
tomorrow's `backward()` will multiply together. (We label the *edges* with operations' local
derivatives rather than drawing separate operation-circles, but the structure is the same DAG.)

In [ ]:
# Optional: graphviz makes prettier pictures. Safe to skip — there is a text fallback.
HAVE_GRAPHVIZ = False
try:
    import graphviz
    HAVE_GRAPHVIZ = True
except ImportError:
    try:
        # On Colab this installs the system tool + the Python binding.
        !apt-get -qq install -y graphviz
        !pip -q install graphviz
        import graphviz
        HAVE_GRAPHVIZ = True
    except Exception as err:
        print("graphviz unavailable — the text fallback will be used (that's fine!).")
        print("  reason:", err)

print("graphviz available:", HAVE_GRAPHVIZ)

In [ ]:
def trace(root):
    """Walk back through ._children, collecting all nodes and edges."""
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._children:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges


def _print_graph(node, depth=0):
    """Plain-text fallback: an indented tree with local grads on the edges."""
    print("    " * depth + f"Value(data={node.data:.4f}, grad={node.grad:.4f})")
    for child, lg in zip(node._children, node._local_grads):
        print("    " * depth + f"    \u2514\u2500 [local grad d(parent)/d(child) = {lg:.4f}]")
        _print_graph(child, depth + 1)


def draw_graph(root):
    """Draw the computation graph as a DAG (graphviz if available, else text)."""
    try:
        from graphviz import Digraph
    except Exception:
        _print_graph(root)
        return None
    nodes, edges = trace(root)
    dot = Digraph(graph_attr={"rankdir": "LR"})   # left-to-right, like the lecture
    for n in nodes:
        dot.node(name=str(id(n)),
                 label="{ data %.3f | grad %.3f }" % (n.data, n.grad),
                 shape="record")
    for child, parent in edges:
        i = parent._children.index(child)         # which child slot this is
        lg = parent._local_grads[i]
        dot.edge(str(id(child)), str(id(parent)), label=" %.3f" % lg)
    return dot

print("Helpers ready: trace(root) and draw_graph(root).")

Now build the lecture's example `c = a * b + 1` and draw it. You should see three leaf
boxes (`a`, `b`, and the constant `1`), a `*` node, and a `+` node — a small DAG.

(Run `draw_graph(c)` as the **last line** of the cell so Colab renders the picture. If graphviz
isn't installed, you'll get the text version instead.)

In [ ]:
a = Value(2.0)
b = Value(3.0)
c = a * b + 1          # the lecture's worked example
print("c =", c, "\n")
draw_graph(c)

## Step 6 — does the forward pass compute the right numbers?

The whole point of the forward pass is to get the **value** right. Let's build
$(2 \times 3 + 1)^2$ with our `Value` class and check it equals plain-Python arithmetic
($7^2 = 49$), plus a few more sanity checks of the convenience operators.

In [ ]:
# Build (2 * 3 + 1) ** 2 with Value and compare to plain Python.
expr = (Value(2.0) * Value(3.0) + 1) ** 2
print("Value result :", expr.data)
print("plain Python :", (2 * 3 + 1) ** 2)
assert expr.data == 49.0, "forward value should be 49"

# A few more forward checks of the convenience operators:
assert (Value(10.0) - Value(4.0)).data == 6.0     # __sub__
assert (Value(8.0)  / Value(2.0)).data == 4.0     # __truediv__  (uses ** -1)
assert (2 + Value(3.0)).data == 5.0               # __radd__
assert (3 * Value(4.0)).data == 12.0              # __rmul__
assert Value(-2.0).relu().data == 0.0             # relu off
assert abs(Value(1.0).exp().data - math.e) < 1e-9 # exp

print("\n\u2705 All forward checks passed \u2014 the graph computes the right numbers.")

## Recap — what you built today

- A `Value` class that **builds a computation graph as you do arithmetic** (`+`, `*`, `**`, `exp`, `log`, `relu`).
- The ⭐ idea: each operation stores its **local derivative** w.r.t. each child in `_local_grads` — e.g. `(b, a)` for multiplication.
- A graph **visualiser** that draws any expression as a DAG, with local gradients on the edges.
- A check that the **forward values are correct** (`(2*3+1)**2 == 49`).

You did *not* write `backward()` — its `.grad` slots are all still `0`. That is on purpose.

### Tomorrow (Lab 4): complete `backward()` and watch it learn

Tomorrow we add one method that walks this graph in reverse, multiplies the local gradients
along the edges (the chain rule), and fills in every `.grad`. With gradients in hand, we can
nudge parameters downhill — and the same `Value` class you built today becomes a tiny neural
network that **learns**. Later, in Lab 11, it scales all the way up to a from-scratch GPT.

Keep this notebook — tomorrow starts exactly where you are now. See you in Lecture 4: *Backpropagation*.

## Solutions

Worked answers for every `# TODO` above. Have a genuine go first — including asking a
model, which will often get you there before this section does.

To pick up where you left off: copy the answer into the matching `# TODO` cell above and
re-run from there, so the rest of the notebook uses your version.

**Solution — the correct `Value` class (forward ops)**

In [ ]:
import math

class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # d(a*b)/da = b, d(a*b)/db = a  -> the swap
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other):
        # power rule: d(x**n)/dx = n * x**(n-1)
        return Value(self.data ** other, (self,), (other * self.data ** (other - 1),))

    def exp(self):
        return Value(math.exp(self.data), (self,), (math.exp(self.data),))

    def log(self):
        return Value(math.log(self.data), (self,), (1 / self.data,))

    def relu(self):
        return Value(max(0, self.data), (self,), (float(self.data > 0),))

    def __neg__(self):             return self * -1
    def __radd__(self, other):     return self + other
    def __sub__(self, other):      return self + (-other)
    def __rsub__(self, other):     return other + (-self)
    def __rmul__(self, other):     return self * other
    def __truediv__(self, other):  return self * other ** -1
    def __rtruediv__(self, other): return other * self ** -1
    def __repr__(self):            return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def backward(self):
        raise NotImplementedError(
            "backward() is built TOMORROW in Lab 4 — that is the step that makes the "
            "network LEARN. Today we only build the forward graph.")

print("Loaded the complete forward Value class. You can keep going.")